# GFED5.1 burned area on the GDSTEM grid

This notebook converts monthly GFED5.1 burned areas on a 0.25-degree grid into
annual burned areas on the aligned 0.5-degree GDSTEM grid, for 2002–2022.
The coordinate CSV supplies TEM lower-left corners; only its lon and lat columns
are used. All 16 GFED classes remain separate. No TEM-community crosswalk is applied.

Calculation steps:

1. Read and deduplicate TEM coordinates, and calculate each full geographic cell area.
2. Find one GFED NetCDF file for each requested year.
3. Read GFED coordinates from the burned-area group or the root group. When an axis
   is absent, use the explicitly configured global axis for that missing axis.
4. Match the four GFED pixel centers inside each TEM cell.
5. For each class and year, sum 12 monthly values and the four source pixels.
   Convert square meters to square kilometers by dividing by 1,000,000.
6. Count valid years and positive-fire years. Calculate burned-area sums, annual
   means, maxima, and mean burned area divided by full TEM cell area.
7. Write the summary table and the sparse annual table described at the end.

For a class c, cell i, and valid year y:

    annual_ba_km2[i, c, y] = sum(monthly_ba_m2 over 12 months and 4 pixels) / 1e6
    mean_ba_km2[i, c] = sum(valid annual_ba_km2) / number_of_valid_years
    mean_grid_fraction[i, c] = mean_ba_km2[i, c] / full_grid_area_km2[i]

A year is valid only when all 48 monthly source values are finite. Valid zero-fire
years count in the mean denominator. Missing data are not treated as zero fire.
Monthly inputs are accumulated areas, so no multiplication by days or seconds is used.
The grid fraction uses the whole geographic cell, including any ocean portion;
it is neither a percentage nor a class-specific burned fraction. It is not clipped to 1.

Input assumptions: 16 variables named BA_01_... through BA_16_..., one 12-slice
monthly dimension per variable, and latitude/longitude dimensions named lat/latitude
and lon/longitude. Source coordinates must describe a regular 0.25-degree grid.
The current input interpretation is monthly accumulated square meters, including
attributes such as "m2 per month". Other units require changing the conversion.
The code checks the number of slices, not their January–December timestamp labels.
Absent coordinates require the configured orientation to be correct; reconstructed
axes cannot prove the true storage orientation. Actual server inputs have not been
verified by this guide.


## Configuration

Edit the input and output paths before running. This workflow interprets source values as monthly accumulated square meters. Run all cells from the beginning after changing the configuration.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# Edit these paths before running on your server.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')
GFED_DIR = Path('/quobyte/emoniergrp/TRENDY/input-files/source-input/GFED5/GFED5.1/Ecosystem')
TEM_GRID_CSV = Path('/quobyte/emoniergrp/emonier/gdstem/gdstem-preprocessing/static-land/static-data/gdstem-potveg-maxcohort.csv')
START_YEAR = 2002
END_YEAR = 2022
N_YEARS = END_YEAR - START_YEAR + 1
GRID_RESOLUTION = 0.5
R_EARTH_KM = 6371.0088

# Used only if the corresponding geographic axis is absent from the file.
# Confirm this orientation for your input files before using reconstructed axes.
MISSING_LATITUDE_ORDER = 'south_to_north'

# Input values are monthly accumulated m2. This is an explicit input assumption.
M2_PER_KM2 = 1000000.0

GFED_NAMES = {
    1: 'Tundra',
    2: 'Sparse boreal forest',
    3: 'Boreal forest',
    4: 'Temperate grassland',
    5: 'Temperate shrubland',
    6: 'Temperate mosaic',
    7: 'Temperate forest',
    8: 'Tropical grassland',
    9: 'Tropical shrubland',
    10: 'Open savanna',
    11: 'Woody savanna',
    12: 'Tropical forest',
    13: 'Other',
    14: 'Cropland',
    15: 'Peat',
    16: 'Deforestation',
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SUMMARY = OUTPUT_DIR / f'02_gfed_by_class_summary_{START_YEAR}_{END_YEAR}.csv'
OUT_ANNUAL = OUTPUT_DIR / f'02_gfed_annual_nonzero_{START_YEAR}_{END_YEAR}.csv'


## TEM grid and annual file list

Read unique TEM lower-left coordinates, calculate full geographic cell areas, and locate exactly one GFED file per requested year.


In [ ]:
# Read only coordinates. Chunking avoids loading the full vegetation table.
grid_parts = []
for chunk in pd.read_csv(TEM_GRID_CSV, usecols=['lon', 'lat'], chunksize=100000):
    grid_parts.append(chunk.drop_duplicates())
grid = pd.concat(grid_parts, ignore_index=True)
grid['lon'] = ((grid['lon'] + 180.0) % 360.0 - 180.0).round(6)
grid['lat'] = grid['lat'].round(6)
grid = grid.drop_duplicates().sort_values(['lat', 'lon']).reset_index(drop=True)

for name in ['lon', 'lat']:
    ticks = grid[name].to_numpy() / GRID_RESOLUTION
    if not np.isfinite(ticks).all() or not np.allclose(ticks, np.round(ticks), rtol=0, atol=1e-6):
        raise ValueError('TEM coordinates must be finite lower-left 0.5-degree ticks.')
if len(grid) == 0 or (grid['lat'] < -90).any() or (grid['lat'] > 89.5).any():
    raise ValueError('The TEM grid is empty or has invalid latitude bounds.')

# Full spherical cell area: R^2 * longitude width * (sin(north) - sin(south)).
south = np.deg2rad(grid['lat'].to_numpy())
north = np.deg2rad(grid['lat'].to_numpy() + GRID_RESOLUTION)
grid['grid_area_km2'] = R_EARTH_KM ** 2 * np.deg2rad(GRID_RESOLUTION) * (np.sin(north) - np.sin(south))
N = len(grid)

files = {}
for year in range(START_YEAR, END_YEAR + 1):
    matches = sorted(GFED_DIR.glob(f'GFED5.1_ecosystem_*{year}.nc'))
    if len(matches) != 1:
        raise ValueError(f'Expected one GFED file for {year}; found {len(matches)}.')
    files[year] = matches[0]
print(f'TEM grid cells: {N:,}; requested years: {N_YEARS}')


## Initialize summary arrays

Prepare one set of summary arrays per GFED class and initialize the annual CSV. Rerun this cell before rerunning the processing cell; otherwise totals would be counted again.


In [ ]:
accum = {}
for class_id in range(1, 17):
    accum[class_id] = {
        'sum': np.zeros(N),
        'max': np.zeros(N),
        'nvalid': np.zeros(N, dtype=int),
        'nfire': np.zeros(N, dtype=int),
    }

# Clear the annual table at the start, including when all valid areas are zero.
# Rerunning the processing cell requires rerunning this initialization cell too.
annual_columns = ['lon', 'lat', 'year', 'gfed_id', 'burned_area_km2']
pd.DataFrame(columns=annual_columns).to_csv(OUT_ANNUAL, index=False)
reference_lon = None
reference_lat = None


## Process years and classes

Read one annual file and one class at a time. Match four source pixels to each TEM cell, sum 12 monthly areas, and update statistics only for valid cell-years. The annual CSV includes only valid positive-fire records.


In [ ]:
for year in range(START_YEAR, END_YEAR + 1):
    print(f'Processing GFED {year}', flush=True)
    with xr.open_dataset(files[year], group='burned_area_partitioning') as ds:
        # Match BA variable names to class IDs once for this year.
        variables = {}
        for name in ds.data_vars:
            parts = name.split('_')
            if len(parts) >= 2 and parts[0] == 'BA' and parts[1].isdigit():
                class_id = int(parts[1])
                if class_id in variables:
                    raise ValueError(f'Duplicate GFED class {class_id} in {year}.')
                variables[class_id] = name
        if sorted(variables) != list(range(1, 17)):
            raise ValueError(f'Expected GFED classes 1 through 16 in {year}.')

        # Determine the spatial dimension names from the first class.
        first = ds[variables[1]]
        lat_dim = 'lat'
        if lat_dim not in first.dims:
            lat_dim = 'latitude'
        lon_dim = 'lon'
        if lon_dim not in first.dims:
            lon_dim = 'longitude'
        nlat = first.sizes[lat_dim]
        nlon = first.sizes[lon_dim]

        # Read group coordinates first, then root coordinates if needed.
        axes = {}
        with xr.open_dataset(files[year]) as root:
            for axis in ['lat', 'lon']:
                if axis == 'lat':
                    possible_names = ['lat', 'latitude']
                else:
                    possible_names = ['lon', 'longitude']
                for dataset in [ds, root]:
                    for name in possible_names:
                        if axis not in axes and name in dataset:
                            axes[axis] = np.asarray(dataset[name].values, dtype=float)

        if 'lat' not in axes:
            if nlat != 720:
                raise ValueError('Reconstructed latitude requires 720 rows.')
            if MISSING_LATITUDE_ORDER == 'south_to_north':
                axes['lat'] = -89.875 + 0.25 * np.arange(nlat)
            elif MISSING_LATITUDE_ORDER == 'north_to_south':
                axes['lat'] = 89.875 - 0.25 * np.arange(nlat)
            else:
                raise ValueError('Set MISSING_LATITUDE_ORDER to a supported orientation.')
            print('Using configured latitude axis:', MISSING_LATITUDE_ORDER)
        if 'lon' not in axes:
            if nlon != 1440:
                raise ValueError('Reconstructed longitude requires 1440 columns.')
            axes['lon'] = -179.875 + 0.25 * np.arange(nlon)
            print('Using configured west-to-east longitude axis.')

        # Reduce separable 2D coordinate arrays to 1D axes when necessary.
        lat = axes['lat']
        lon = axes['lon']
        if lat.ndim == 2 and lat.shape == (nlat, nlon):
            if np.allclose(lat, lat[:, :1], rtol=0, atol=1e-6):
                lat = lat[:, 0]
        if lon.ndim == 2 and lon.shape == (nlat, nlon):
            if np.allclose(lon, lon[:1, :], rtol=0, atol=1e-6):
                lon = lon[0, :]
        lon = (lon + 180.0) % 360.0 - 180.0
        for values, count in [(lat, nlat), (lon, nlon)]:
            if values.ndim != 1 or len(values) != count or not np.isfinite(values).all():
                raise ValueError('Expected finite 1D latitude and longitude axes.')
            if not np.allclose(np.diff(np.sort(values)), 0.25, rtol=0, atol=1e-6):
                raise ValueError('Expected regular 0.25-degree coordinate spacing.')
        if lat.min() < -90 or lat.max() > 90:
            raise ValueError('Latitude coordinates are outside geographic bounds.')

        if reference_lon is None:
            reference_lon = lon.copy()
            reference_lat = lat.copy()

            # Match source centers to the TEM lower-left coordinates.
            lon_lower = np.round(np.floor(lon / 0.5 + 1e-10) * 0.5, 6)
            lat_lower = np.round(np.floor(lat / 0.5 + 1e-10) * 0.5, 6)
            source_lon, source_lat = np.meshgrid(lon_lower, lat_lower)
            source_table = pd.DataFrame({
                'lon': source_lon.ravel(),
                'lat': source_lat.ravel(),
                'source_index': np.arange(nlat * nlon),
            })
            target_table = grid[['lon', 'lat']].copy()
            target_table['target_id'] = np.arange(N)
            mapping = source_table.merge(target_table, on=['lon', 'lat'], how='inner')
            source_counts = mapping.groupby('target_id').size().reindex(range(N), fill_value=0)
            if (source_counts != 4).any():
                raise ValueError('Each TEM cell must match exactly four GFED pixels.')
            source_indices = mapping['source_index'].to_numpy()
            target_ids = mapping['target_id'].to_numpy()
        else:
            if not np.array_equal(lon, reference_lon) or not np.array_equal(lat, reference_lat):
                raise ValueError(f'GFED coordinates changed in {year}.')

        annual_rows = []
        for class_id in range(1, 17):
            da = ds[variables[class_id]]
            month_dims = []
            for dim in da.dims:
                if dim != lat_dim and dim != lon_dim:
                    month_dims.append(dim)
            if len(month_dims) != 1 or da.sizes[month_dims[0]] != 12:
                raise ValueError(f'Expected 12 monthly slices for class {class_id} in {year}.')
            if da.sizes[lat_dim] != nlat or da.sizes[lon_dim] != nlon:
                raise ValueError('GFED classes must share the same spatial grid.')

            # Load one class at a time in month-latitude-longitude order.
            monthly = np.asarray(da.transpose(month_dims[0], lat_dim, lon_dim).values, dtype=float)
            monthly = monthly.reshape(12, -1)[:, source_indices]
            valid_source = np.isfinite(monthly).all(axis=0)
            if np.any((monthly < -1e-9) & np.isfinite(monthly)):
                raise ValueError(f'Negative burned area for class {class_id} in {year}.')

            # Sum monthly accumulated m2, then convert to annual km2.
            annual_source = monthly.sum(axis=0) / M2_PER_KM2
            source_values = pd.DataFrame({
                'target_id': target_ids,
                'area': annual_source,
                'valid': valid_source.astype(int),
            })
            totals = source_values.groupby('target_id')[['area', 'valid']].sum()
            totals = totals.reindex(range(N))
            annual_area = totals['area'].to_numpy()
            valid_cells = totals['valid'].to_numpy() == 4
            positive_cells = valid_cells & (annual_area > 0)

            # Invalid cell-years do not update any summary statistic.
            state = accum[class_id]
            state['sum'][valid_cells] += annual_area[valid_cells]
            state['max'][valid_cells] = np.maximum(state['max'][valid_cells], annual_area[valid_cells])
            state['nvalid'] += valid_cells.astype(int)
            state['nfire'] += positive_cells.astype(int)

            if positive_cells.any():
                rows = grid.loc[positive_cells, ['lon', 'lat']].copy()
                rows['year'] = year
                rows['gfed_id'] = class_id
                rows['burned_area_km2'] = annual_area[positive_cells]
                annual_rows.append(rows)

        if annual_rows:
            pd.concat(annual_rows, ignore_index=True).to_csv(
                OUT_ANNUAL, mode='a', header=False, index=False
            )


## Write summaries

Calculate means using valid-year counts, including valid zero-fire years. Write one row per TEM cell and GFED class. Use the output files only after all cells finish successfully.


In [ ]:
for class_id in range(1, 17):
    state = accum[class_id]
    valid_years = state['nvalid']
    has_data = valid_years > 0
    mean_area = np.full(N, np.nan)
    max_area = np.full(N, np.nan)
    mean_area[has_data] = state['sum'][has_data] / valid_years[has_data]
    max_area[has_data] = state['max'][has_data]

    result = grid[['lon', 'lat', 'grid_area_km2']].copy()
    result['gfed_id'] = class_id
    result['gfed_name'] = GFED_NAMES[class_id]
    result['gfed_ba_sum_km2'] = state['sum']
    result['gfed_mean_ba_km2'] = mean_area
    result['gfed_max_annual_ba_km2'] = max_area
    result['gfed_valid_years'] = valid_years
    result['gfed_nonzero_years'] = state['nfire']
    result['gfed_mean_grid_fraction'] = mean_area / grid['grid_area_km2'].to_numpy()

    if class_id == 1:
        result.to_csv(OUT_SUMMARY, index=False)
    else:
        result.to_csv(OUT_SUMMARY, mode='a', header=False, index=False)

print('Saved:', OUT_SUMMARY)
print('Saved:', OUT_ANNUAL)


Both files are written to OUTPUT_DIR. The filenames below apply to 2002–2022 and
change automatically when START_YEAR or END_YEAR changes. Existing output files
are overwritten when the corresponding initialization or writing cells run.
A stopped run can leave incomplete output; use the files only after the full run finishes.

### 02_gfed_by_class_summary_2002_2022.csv

One row per TEM cell and GFED class, including classes with no valid fire records.
Expected row count: number of unique TEM cells multiplied by 16.

| Column | Meaning |
| --- | --- |
| lon, lat | TEM lower-left coordinates in degrees |
| grid_area_km2 | Full geographic 0.5-degree cell area |
| gfed_id, gfed_name | GFED class identifier and name; not a TEM community |
| gfed_ba_sum_km2 | Sum of annual burned areas over valid years |
| gfed_mean_ba_km2 | Sum divided by valid-year count, including valid zero-fire years |
| gfed_max_annual_ba_km2 | Maximum annual burned area over valid years |
| gfed_valid_years | Years with all four pixels and all 12 months finite |
| gfed_nonzero_years | Valid years with strictly positive annual burned area |
| gfed_mean_grid_fraction | Mean annual burned area divided by full geographic cell area |

Use this table for later class-to-community matching, fire-activity grouping,
and completeness filtering. For the complete 2002–2022 period, require
`gfed_valid_years == 21` before interpreting the mean as a complete-period mean.
With zero valid years, sums and counts are zero, while means, maxima and fractions
are missing. A zero sum alone does not establish a valid zero-fire record.

### 02_gfed_annual_nonzero_2002_2022.csv

One row per valid positive-fire cell, class and year. Columns:
`lon`, `lat`, `year`, `gfed_id`, `burned_area_km2`.

Use this table to inspect fire timing, compare positive annual burned areas,
or retrieve annual records for selected cells. It omits valid zero-fire years
and invalid years. A missing row therefore does not establish zero fire.
The summary table reports how many years are valid, but does not identify which
omitted years are invalid. These two outputs alone cannot reconstruct a fully
labelled annual zero-versus-missing time series for incomplete records.
No additional audit tables or intermediate mapping files are written.
